# Week 3 — The Forward Diffusion Process (Reference Solution)

> **Mentor reference.** Keep private until after the Week 3 deadline.

Implements the **forward diffusion process**: the recipe for gradually turning a clean image into pure Gaussian noise, and the closed-form shortcut that lets us jump to any timestep `t` in one step. This is the mathematical foundation for Week 4's training.

No neural network this week — just the scheduler and visualizations.

**Runtime:** seconds (CPU is fine).

In [ ]:
import torch, math
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
torch.manual_seed(0)

## 1. Noise schedules

The schedule defines how much noise is added at each step via the variances $\beta_t$. From the betas we derive:
- $\alpha_t = 1 - \beta_t$
- $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ (cumulative product)

The cumulative product $\bar{\alpha}_t$ is the key quantity: it lets us sample $x_t$ directly from $x_0$ without iterating.

In [ ]:
def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

def cosine_beta_schedule(T, s=0.008):
    # Nichol & Dhariwal 2021 — smoother, better-conditioned near t=T
    steps = T + 1
    x = torch.linspace(0, T, steps)
    acp = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
    acp = acp / acp[0]
    betas = 1 - (acp[1:] / acp[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

## 2. The `NoiseScheduler`

In [ ]:
class NoiseScheduler:
    def __init__(self, T=1000, schedule="linear"):
        self.T = T
        self.betas = linear_beta_schedule(T) if schedule == "linear" else cosine_beta_schedule(T)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_acp = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_acp = torch.sqrt(1.0 - self.alphas_cumprod)

    def q_sample(self, x0, t, noise=None):
        """Sample x_t ~ q(x_t | x_0) in closed form (the reparameterization trick).
        x_t = sqrt(acp_t) * x0 + sqrt(1 - acp_t) * noise"""
        if noise is None:
            noise = torch.randn_like(x0)
        a = self.sqrt_acp[t].view(-1, 1, 1, 1)
        b = self.sqrt_one_minus_acp[t].view(-1, 1, 1, 1)
        return a * x0 + b * noise

sched = NoiseScheduler(T=1000, schedule="linear")
print("betas:", sched.betas[:3], "...", sched.betas[-1].item())
print("alphas_cumprod[0]:", sched.alphas_cumprod[0].item(),
      " [-1]:", sched.alphas_cumprod[-1].item())

## 3. Verify the endpoint is pure noise

If the schedule is correct, by $t = T$ the image should be indistinguishable from $\mathcal{N}(0, I)$: mean ≈ 0, std ≈ 1.

In [ ]:
x0 = torch.randn(256, 1, 28, 28)        # stand-in batch; real images give the same result
tT = torch.full((256,), sched.T - 1, dtype=torch.long)
xT = sched.q_sample(x0, tT)
print(f"x_T mean = {xT.mean():.4f}  (want ~0.0)")
print(f"x_T std  = {xT.std():.4f}   (want ~1.0)")
assert abs(xT.mean()) < 0.1 and abs(xT.std() - 1.0) < 0.1
print("Forward process endpoint is approximately N(0, I). Correct.")

## 4. Visualize the noising trajectory on a real image

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist = datasets.MNIST("./data", train=True, download=True, transform=tf)
img, _ = mnist[0]
img = img.unsqueeze(0)  # (1,1,28,28)

steps_to_show = [0, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(steps_to_show), figsize=(13, 2.4))
for ax, t in zip(axes, steps_to_show):
    t_tensor = torch.tensor([t])
    noised = sched.q_sample(img, t_tensor, noise=torch.randn_like(img))
    ax.imshow(((noised[0, 0].clamp(-1, 1) + 1) / 2), cmap="gray")
    ax.set_title(f"t={t}"); ax.axis("off")
plt.suptitle("Forward diffusion: clean image (left) to pure noise (right)")
plt.tight_layout(); plt.show()

## 5. Linear vs cosine: signal-to-noise ratio

In [ ]:
lin = NoiseScheduler(1000, "linear")
cos = NoiseScheduler(1000, "cosine")
# SNR(t) = acp / (1 - acp)
snr_lin = lin.alphas_cumprod / (1 - lin.alphas_cumprod)
snr_cos = cos.alphas_cumprod / (1 - cos.alphas_cumprod)
plt.figure(figsize=(8, 4))
plt.plot(snr_lin.log10(), label="linear")
plt.plot(snr_cos.log10(), label="cosine")
plt.xlabel("timestep t"); plt.ylabel("log10 SNR"); plt.legend()
plt.title("Signal-to-noise ratio: cosine retains signal longer")
plt.show()

## Self-check answers (for mentor)

1. **Why sample $x_t$ in one step?** Because the composition of Gaussian noising steps is itself Gaussian with a closed form: $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$. Without it, training would need a `t`-length loop per sample.
2. **$\beta$ vs $\alpha$ vs $\bar\alpha$?** $\beta_t$ = noise variance added at step `t`; $\alpha_t = 1-\beta_t$; $\bar\alpha_t$ = cumulative product, the total signal retained from $x_0$ to step `t`.
3. **Why a schedule at all?** Adding noise gradually makes the reverse (denoising) steps small and learnable. Jumping straight to full noise leaves nothing to learn.
4. **Reparameterization trick?** Writing $x_t$ as a deterministic function of $x_0$ and a sampled $\epsilon$, so gradients flow and we can train the network to predict $\epsilon$.
5. **Why cosine > linear?** Linear destroys signal too quickly at high `t`; cosine keeps a useful SNR longer, improving sample quality.

## Common mentee mistakes
- Confusing $\alpha$ with $\bar\alpha$.
- Forgetting to normalize images to $[-1, 1]$ first.
- Numerical issues taking `sqrt` of tiny $\bar\alpha$ near `t=T` (clip betas).